# Proyecto Final: Q-Learning para Resolución de Laberintos

**Autores:** Nicolas Steven Lara Villa y Jersson Mauricio Rodríguez Aponte  
**Fecha:** Noviembre 2025  
**Objetivo:** Implementar un agente de Reinforcement Learning (Q-Learning) capaz de resolver laberintos representados como cuadrículas con paredes.

---

## **Instrucciones para utilizar la Q-Tabla**

Siga estos pasos para ejecutar correctamente el script:

1️⃣ Ubicar la Q-Tabla: Si ya cuenta con una **Q-Tabla** en formato `.npy`, colóquela en la carpeta raíz del proyecto.

2️⃣ Configurar el nombre del archivo: En el código, asigne el nombre del archivo a la variable ``ARCHIVO_Q_TABLE = "q_table.npy"``.
Asegúrese de que el nombre coincida exactamente con el archivo ubicado en la raíz.

3️⃣ Ejecutar el proyecto: Ejecute todo el notebook usando *Run All*.

* Si la Q-Tabla es válida: El sistema cargará este archivo y lo utilizará directamente para evaluar el agente.

* Si la Q-Tabla es inválida o no existe: El sistema entrenará automáticamente un nuevo agente, generará un archivo .npy con la nueva Q-Tabla en la raíz y procederá a usarla para la evaluación.
---

In [ ]:
# ----------------------------------------------------
#  PASO 1: Defina el nombre del archivo para cargar la Q-tabla. Asegúrese de que exista en la carpeta raiz del proyecto.
# ----------------------------------------------------

ARCHIVO_Q_TABLE = "q_table.npy" # Ejemplo: "q_table.npy"

# Si desea entrenar un nuevo agente, asigne un nombre nuevo a la Q-tabla. El script entrenará y guardará la Q-tabla con ese nombre.

# ----------------------------------------------------
#  PASO 2: Ejecute todo el notebook para cargar la Q-tabla y evaluar el agente
# ----------------------------------------------------

In [ ]:

import os

def validar_q_table (archivo_q_table: str) -> bool:
    """
    Valida si existe un archivo de Q-tabla guardado previamente.
    
    Args:
        archivo_q_table: Nombre del archivo de Q-tabla a validar
        
    Returns:
        True si el archivo existe y tiene contenido válido, False en caso contrario
    """
    # Verificar que el parámetro no esté vacío
    if not archivo_q_table or archivo_q_table.strip() == "":
        print("⚠️  ARCHIVO_Q_TABLE no tiene nombre asignado")
        return False
    
    # Verificar si el archivo existe
    if not os.path.exists(archivo_q_table):
        print(f"⚠️  Archivo '{archivo_q_table}' no encontrado")
        return False
    
    # Verificar que sea un archivo (no directorio)
    if not os.path.isfile(archivo_q_table):
        print(f"⚠️  '{archivo_q_table}' no es un archivo válido")
        return False
    
    # Verificar que el archivo tenga contenido
    if os.path.getsize(archivo_q_table) == 0:
        print(f"⚠️  Archivo '{archivo_q_table}' está vacío")
        return False
    
    print(f"✅ Archivo '{archivo_q_table}' encontrado y válido")
    return True

# =============================================================================
# VALIDACIÓN DE Q-TABLA EXISTENTE
# =============================================================================

VALIDAR_Q_TABLA = validar_q_table(ARCHIVO_Q_TABLE)

if VALIDAR_Q_TABLA:
    print(f"\n🎯 Estado: Q-tabla encontrada - Se carga para evaluación")
else:
    print(f"\n📝 Estado: Q-tabla no encontrada - Se entrenará un nuevo agente")

print(f"   Variable VALIDAR_Q_TABLA = {VALIDAR_Q_TABLA}")

# **1. HIPERPARÁMETROS Y CONFIGURACIÓN**

Centralizamos todos los parámetros del sistema aquí para facilitar experimentación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from typing import Tuple, List, Set, Optional
import random
import time

# =============================================================================
# HIPERPARÁMETROS DE Q-LEARNING
# =============================================================================

# Parámetros del algoritmo
TASA_APRENDIZAJE = 0.1          # α (alpha): controla cuánto actualizamos Q en cada paso
FACTOR_DESCUENTO = 0.95         # γ (gamma): importancia de recompensas futuras (0=miope, 1=previsor)

# Parámetros de exploración (ε-greedy con decay)
EPSILON_INICIAL = 1.0           # Comenzamos explorando al 100%
EPSILON_MINIMO = 0.01           # No bajamos de 1% de exploración
EPSILON_DECAY = 0.998           # Decaimiento multiplicativo por episodio: ε_new = ε_old * decay

# Parámetros de entrenamiento
NUM_EPISODIOS = 4000            # Número de episodios de entrenamiento
MAX_PASOS_POR_EPISODIO = 200    # Límite de pasos por episodio (2*n*m sería típico para 8x7)

# Recompensas
REWARD_GOAL = 100.0             # Recompensa por alcanzar la meta
REWARD_STEP = -1.0              # Penalización por cada paso (favorece rutas cortas)
REWARD_WALL = -10.0             # Penalización por chocar contra pared
REWARD_TIMEOUT = -50.0          # Penalización adicional por exceder MAX_PASOS

# Configuración general
RANDOM_SEED = 42                # Semilla para reproducibilidad
ARCHIVO_LABERINTO = "laberinto.txt"

# Acciones posibles (movimientos cardinales)
ACCIONES = {
    0: "ARRIBA",     # Δfila = -1, Δcol = 0
    1: "ABAJO",      # Δfila = +1, Δcol = 0
    2: "IZQUIERDA",  # Δfila = 0, Δcol = -1
    3: "DERECHA"     # Δfila = 0, Δcol = +1
}
NUM_ACCIONES = len(ACCIONES)

# Deltas de movimiento para cada acción
DELTAS_ACCION = {
    0: (-1, 0),   # ARRIBA
    1: (1, 0),    # ABAJO
    2: (0, -1),   # IZQUIERDA
    3: (0, 1)     # DERECHA
}

# Configurar semilla para reproducibilidad
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("✅ Hiperparámetros configurados correctamente")
print(f"   - Tasa de aprendizaje (α): {TASA_APRENDIZAJE}")
print(f"   - Factor de descuento (γ): {FACTOR_DESCUENTO}")
print(f"   - Epsilon inicial: {EPSILON_INICIAL}")
print(f"   - Episodios de entrenamiento: {NUM_EPISODIOS}")

# **2. IDENTIFICACIÓN DEL LABERINTO, DEFINICIÓN DEL AMBIENTE Y EL AGENTE**

## Identificación y mapeo del laberinto

Primero necesitamos leer y parsear el archivo `.txt` que describe el laberinto.

**Formato del archivo:**
- Línea 1: `n m` (filas y columnas)
- Línea 2: `k` (número de segmentos de pared)
- Siguientes k líneas: `x1 y1 x2 y2` (coordenadas de cada segmento de pared)

In [ ]:
def parse_maze_file(filename: str) -> Tuple[int, int, List[Tuple[int, int, int, int]]]:
    """
    Lee y parsea el archivo de laberinto.

    Formato:
    - Línea 1: n_filas n_columnas
    - Línea 2: k (número de segmentos)
    - Siguientes k líneas: x1 y1 x2 y2 (coordenadas de esquinas)
    """
    with open(filename, 'r') as f:
        lines = [l.strip() for l in f.readlines()]
    
    n_filas, n_columnas = map(int, lines[0].split())
    k = int(lines[1])
    
    segmentos_pared = []
    for i in range(2, 2 + k):
        x1, y1, x2, y2 = map(int, lines[i].split())
        segmentos_pared.append((x1, y1, x2, y2))
    
    return n_filas, n_columnas, segmentos_pared

def construir_conjunto_paredes(
    segmentos: List[Tuple[int, int, int, int]],
    n_filas: int,
    n_columnas: int
) -> Set[Tuple[Tuple[int, int], Tuple[int, int]]]:
    """
    Construye un conjunto de bordes donde existen paredes.

    Cada borde se representa como una tupla de dos celdas adyacentes:
    ((fila1, col1), (fila2, col2)) ordenadas lexicográficamente.
    """
    paredes: Set[Tuple[Tuple[int, int], Tuple[int, int]]] = set()

    for x1, y1, x2, y2 in segmentos:

        # ----------------------
        # PARED HORIZONTAL
        # ----------------------
        # Segmento entre (x, y1) y (x, y2) → misma fila, separa ARRIBA / ABAJO
        if x1 == x2:
            fila = x1
            col_min, col_max = sorted((y1, y2))

            for col in range(col_min, col_max):
                celda_arriba = (fila - 1, col)
                celda_abajo  = (fila, col)

                if (0 <= celda_arriba[0] < n_filas and
                    0 <= celda_abajo[0]  < n_filas and
                    0 <= col < n_columnas):

                    borde = tuple(sorted((celda_arriba, celda_abajo)))
                    paredes.add(borde)

        # ----------------------
        # PARED VERTICAL
        # ----------------------
        # Segmento entre (x1, y) y (x2, y) → misma columna, separa IZQ / DER
        elif y1 == y2:
            col = y1
            fila_min, fila_max = sorted((x1, x2))

            for fila in range(fila_min, fila_max):
                celda_izq = (fila, col - 1)
                celda_der = (fila, col)

                if (0 <= fila < n_filas and
                    0 <= celda_izq[1] < n_columnas and
                    0 <= celda_der[1] < n_columnas):

                    borde = tuple(sorted((celda_izq, celda_der)))
                    paredes.add(borde)

    return paredes

def hay_pared_entre(
    celda1: Tuple[int, int],
    celda2: Tuple[int, int],
    paredes: Set[Tuple[Tuple[int, int], Tuple[int, int]]]
) -> bool:
    """
    Verifica si existe una pared entre dos celdas adyacentes.
    """
    borde = (celda1, celda2) if celda1 < celda2 else (celda2, celda1)
    return borde in paredes

def encontrar_start(
    n_filas: int,
    n_columnas: int,
    segmentos: List[Tuple[int, int, int, int]]
) -> Tuple[int, int]:
    """
    Encuentra START: primera celda en columna 0 sin pared en el borde izquierdo.
    El borde izquierdo es la línea vertical y=0.
    """
    filas_bloqueadas = set()

    for x1, y1, x2, y2 in segmentos:
        # Segmento vertical en y=0
        if y1 == 0 and y2 == 0:
            fila_min, fila_max = sorted((x1, x2))
            for fila in range(fila_min, fila_max):
                if 0 <= fila < n_filas:
                    filas_bloqueadas.add(fila)

    for fila in range(n_filas):
        if fila not in filas_bloqueadas:
            return (fila, 0)

    raise ValueError("No se encontró estado START válido.")


def encontrar_goal(
    n_filas: int,
    n_columnas: int,
    segmentos: List[Tuple[int, int, int, int]]
) -> Tuple[int, int]:
    """
    Encuentra GOAL: primera celda en última columna sin pared en el borde derecho.
    El borde derecho es la línea vertical y = n_columnas.
    """
    filas_bloqueadas = set()

    for x1, y1, x2, y2 in segmentos:
        # Segmento vertical en y = n_columnas
        if y1 == n_columnas and y2 == n_columnas:
            fila_min, fila_max = sorted((x1, x2))
            for fila in range(fila_min, fila_max):
                if 0 <= fila < n_filas:
                    filas_bloqueadas.add(fila)

    for fila in range(n_filas):
        if fila not in filas_bloqueadas:
            return (fila, n_columnas - 1)

    raise ValueError("No se encontró estado GOAL válido.")

def renderizar_laberinto(
    filename: str,
    mostrar_indices: bool = True,
    tamaño: int = 6
):
    """
    Renderiza el laberinto definido en el archivo .txt
    usando el sistema de coordenadas del proyecto.
    """
    # 1. Parsear archivo y detectar estados clave
    n_filas, n_columnas, segmentos = parse_maze_file(filename)
    start = encontrar_start(n_filas, n_columnas, segmentos)
    goal  = encontrar_goal(n_filas, n_columnas, segmentos)

    fig, ax = plt.subplots(figsize=(tamaño, tamaño))

    # 2. Dibujar celdas
    for fila in range(n_filas):
        for col in range(n_columnas):
            ax.add_patch(
                plt.Rectangle(
                    (col, fila),  # (x, y) = (columna, fila)
                    1, 1,
                    edgecolor="lightgray",
                    facecolor="white",
                    linewidth=0.5
                )
            )

    # 3. Dibujar paredes a partir de los segmentos
    for x1, y1, x2, y2 in segmentos:
        # x = fila, y = columna → mapear a (col, fila)
        x_plot1, y_plot1 = y1, x1
        x_plot2, y_plot2 = y2, x2
        ax.plot([x_plot1, x_plot2], [y_plot1, y_plot2], color="black", linewidth=2)

    # 4. Marcar START y GOAL
    sf, sc = start
    gf, gc = goal

    ax.add_patch(
        plt.Rectangle(
            (sc, sf),
            1, 1,
            facecolor="red",
            alpha=0.6
        )
    )
    ax.add_patch(
        plt.Rectangle(
            (gc, gf),
            1, 1,
            facecolor="blue",
            alpha=0.6
        )
    )

    # 5. Ajustes visuales
    ax.set_xlim(0, n_columnas)
    ax.set_ylim(0, n_filas)
    ax.set_aspect("equal")
    ax.invert_yaxis()  # fila 0 arriba, como en la imagen del proyecto

    ax.set_xticks(range(n_columnas + 1))
    ax.set_yticks(range(n_filas + 1))

    if mostrar_indices:
        ax.set_xticklabels(range(n_columnas + 1))
        ax.set_yticklabels(range(n_filas + 1))
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    plt.title("Laberinto")
    plt.grid(False)
    plt.show()

### Renderizar laberinto

In [ ]:
# ======================================================
# 🚀 PARSEO DEL LABERINTO
# ======================================================
n, m, segmentos = parse_maze_file(ARCHIVO_LABERINTO)

print("\n" + "="*60)
print("📄 RESULTADO DEL PARSEO DEL LABERINTO")
print("="*60)
print(f"➡️  Dimensiones del laberinto : {n} filas × {m} columnas")
print(f"➡️  Total de celdas           : {n * m}")
print(f"➡️  Segmentos de pared         : {len(segmentos)}")
print(f"➡️  Primeros 5 segmentos       : {segmentos[:5]}")

# Construir paredes reales entre celdas
paredes = construir_conjunto_paredes(segmentos, n, m)

print("\n" + "="*60)
print("🧱 PAREDES INTERPRETADAS")
print("="*60)
print(f"➡️  Total de paredes entre celdas: {len(paredes)}")
print(f"➡️  Ejemplos de paredes: {list(paredes)[:5]}")

# ======================================================
# 🎯 DETECCIÓN AUTOMÁTICA DE START Y GOAL
# ======================================================
start_state = encontrar_start(n, m, segmentos)
goal_state = encontrar_goal(n, m, segmentos)

distancia = abs(goal_state[0] - start_state[0]) + abs(goal_state[1] - start_state[1])

print("\n" + "="*60)
print("🎯 ESTADOS CLAVE DEL LABERINTO")
print("="*60)
print(f"🟩 START : {start_state}   (fila={start_state[0]}, col={start_state[1]})")
print(f"🟥 GOAL  : {goal_state}    (fila={goal_state[0]}, col={goal_state[1]})")

# ======================================================
# 🖼️ RENDER DEL LABERINTO
# ======================================================
print("\n" + "="*60)
print("🖼️ REPRESENTACIÓN VISUAL DEL LABERINTO")
print("="*60)
renderizar_laberinto(ARCHIVO_LABERINTO)


## **Definición del ambiente**

### Clase MazeEnv - Ambiente del Laberinto

Esta clase implementa el **ambiente** siguiendo el principio de **Responsabilidad Única**: solo maneja la lógica del laberinto, NO el aprendizaje.

**Interfaz estándar de RL:**
- `reset()` → estado_inicial
- `step(action)` → (next_state, reward, done, info)
- `render()` → visualización

In [ ]:
class MazeEnv:
    """
    Ambiente de laberinto para Reinforcement Learning.
    
    Implementa la interfaz estándar de RL con métodos reset(), step() y render().
    Principio de Responsabilidad Única: solo maneja el ambiente, no el aprendizaje.
    """
    
    def __init__(self, archivo_laberinto: str):
        """
        Inicializa el ambiente del laberinto.
        
        Args:
            archivo_laberinto: Ruta al archivo .txt con el laberinto
        """
        # Parsear archivo
        self.n_filas, self.n_columnas, segmentos = parse_maze_file(archivo_laberinto)
        
        # Construir estructura de paredes
        self.paredes = construir_conjunto_paredes(segmentos, self.n_filas, self.n_columnas)
        
        # Detectar estados especiales
        self.start_state = encontrar_start(self.n_filas, self.n_columnas, segmentos)
        self.goal_state = encontrar_goal(self.n_filas, self.n_columnas, segmentos)
        
        # Estado actual (se inicializa en reset)
        self.current_state = None
        self.num_steps = 0
        
        # Espacio de estados
        self.num_states = self.n_filas * self.n_columnas
        
        print(f"🎮 Ambiente MazeEnv inicializado:")
        print(f"   - Dimensiones: {self.n_filas}×{self.n_columnas} ({self.num_states} estados)")
        print(f"   - START: {self.start_state}")
        print(f"   - GOAL: {self.goal_state}")
    
    def reset(self) -> Tuple[int, int]:
        """
        Reinicia el ambiente al estado inicial.
        
        Returns:
            Estado inicial (START)
        """
        self.current_state = self.start_state
        self.num_steps = 0
        return self.current_state
    
    def state_to_index(self, state: Tuple[int, int]) -> int:
        """
        Convierte un estado (fila, col) a un índice lineal para la Q-tabla.
        
        Args:
            state: Estado como (fila, columna)
            
        Returns:
            Índice lineal en [0, num_states-1]
        """
        fila, col = state
        return fila * self.n_columnas + col
    
    def index_to_state(self, index: int) -> Tuple[int, int]:
        """
        Convierte un índice lineal a un estado (fila, col).
        
        Args:
            index: Índice lineal
            
        Returns:
            Estado como (fila, columna)
        """
        fila = index // self.n_columnas
        col = index % self.n_columnas
        return (fila, col)
    
    def is_valid_action(self, state: Tuple[int, int], action: int) -> bool:
        """
        Verifica si una acción es válida desde un estado dado.
        
        Una acción es válida si:
        1. La nueva celda está dentro de los límites del tablero
        2. No hay pared entre la celda actual y la nueva celda
        
        Args:
            state: Estado actual (fila, columna)
            action: Acción a validar (0=ARRIBA, 1=ABAJO, 2=IZQ, 3=DER)
            
        Returns:
            True si la acción es válida, False si no
        """
        fila, col = state
        delta_fila, delta_col = DELTAS_ACCION[action]
        
        nueva_fila = fila + delta_fila
        nueva_col = col + delta_col
        
        # Verificar límites del tablero
        if not (0 <= nueva_fila < self.n_filas and 0 <= nueva_col < self.n_columnas):
            return False
        
        # Verificar si hay pared entre celdas
        nueva_celda = (nueva_fila, nueva_col)
        if hay_pared_entre(state, nueva_celda, self.paredes):
            return False
        
        return True
    
    def step(self, action: int) -> Tuple[Tuple[int, int], float, bool, dict]:
        """
        Ejecuta una acción en el ambiente.
        
        Args:
            action: Acción a ejecutar (0=ARRIBA, 1=ABAJO, 2=IZQ, 3=DER)
            
        Returns:
            Tupla (next_state, reward, done, info) donde:
            - next_state: Nuevo estado después de la acción
            - reward: Recompensa obtenida
            - done: True si el episodio terminó (alcanzó GOAL o timeout)
            - info: Diccionario con información adicional
        """
        self.num_steps += 1
        info = {"action_name": ACCIONES[action], "valid_action": True}
        
        # Verificar si la acción es válida
        if not self.is_valid_action(self.current_state, action):
            # Acción inválida: penalización y el agente se queda en el mismo lugar
            reward = REWARD_WALL
            info["valid_action"] = False
            return self.current_state, reward, False, info
        
        # Ejecutar acción válida
        fila, col = self.current_state
        delta_fila, delta_col = DELTAS_ACCION[action]
        next_state = (fila + delta_fila, col + delta_col)
        
        # Actualizar estado actual
        self.current_state = next_state
        
        # Calcular recompensa
        if next_state == self.goal_state:
            # ¡Alcanzó la meta!
            reward = REWARD_GOAL
            done = True
            info["reason"] = "goal_reached"
        elif self.num_steps >= MAX_PASOS_POR_EPISODIO:
            # Timeout: excedió el límite de pasos
            reward = REWARD_STEP + REWARD_TIMEOUT
            done = True
            info["reason"] = "timeout"
        else:
            # Paso normal
            reward = REWARD_STEP
            done = False
        
        return next_state, reward, done, info
    
    def render(self, agent_path: Optional[List[Tuple[int, int]]] = None, 
               show_grid: bool = True):
        """
        Visualiza el laberinto con matplotlib.
        
        Args:
            agent_path: Lista de estados visitados por el agente (opcional)
            show_grid: Si True, muestra la grilla de celdas
        """
        fig, ax = plt.subplots(figsize=(10, 10))
        
        # Dibujar paredes
        for (celda1, celda2) in self.paredes:
            f1, c1 = celda1
            f2, c2 = celda2
            
            # Calcular coordenadas del segmento de pared
            if f1 == f2:  # Pared vertical (separa en horizontal)
                x = [max(c1, c2), max(c1, c2)]
                y = [f1, f1 + 1]
            else:  # Pared horizontal (separa en vertical)
                x = [c1, c1 + 1]
                y = [max(f1, f2), max(f1, f2)]
            
            ax.plot(x, y, 'k-', linewidth=2)
        
        # Dibujar bordes exteriores del laberinto
        ax.plot([0, self.n_columnas], [0, 0], 'k-', linewidth=3)
        ax.plot([0, self.n_columnas], [self.n_filas, self.n_filas], 'k-', linewidth=3)
        ax.plot([0, 0], [0, self.n_filas], 'k-', linewidth=3)
        ax.plot([self.n_columnas, self.n_columnas], [0, self.n_filas], 'k-', linewidth=3)
        
        # Dibujar estado START
        start_f, start_c = self.start_state
        ax.plot(start_c + 0.5, start_f + 0.5, 'ro', markersize=20, label='START')
        
        # Dibujar estado GOAL
        goal_f, goal_c = self.goal_state
        ax.plot(goal_c + 0.5, goal_f + 0.5, 'bo', markersize=20, label='GOAL')
        
        # Dibujar trayectoria del agente si se proporciona
        if agent_path is not None and len(agent_path) > 0:
            path_x = [c + 0.5 for f, c in agent_path]
            path_y = [f + 0.5 for f, c in agent_path]
            ax.plot(path_x, path_y, 'g-', linewidth=2, alpha=0.6, label='Trayectoria')
            
            # Marcar posición actual
            if len(agent_path) > 0:
                curr_f, curr_c = agent_path[-1]
                ax.plot(curr_c + 0.5, curr_f + 0.5, 'gs', markersize=15, label='Agente')
        
        # Configuración de visualización
        ax.set_xlim(0, self.n_columnas)
        ax.set_ylim(0, self.n_filas)
        ax.set_aspect('equal')
        ax.invert_yaxis()  # Invertir eje Y para que (0,0) esté arriba-izquierda
        ax.set_xlabel('Columnas (Y)', fontsize=12)
        ax.set_ylabel('Filas (X)', fontsize=12)
        ax.set_title('Laberinto', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right')
        
        if show_grid:
            ax.grid(True, alpha=0.3)
            ax.set_xticks(range(self.n_columnas + 1))
            ax.set_yticks(range(self.n_filas + 1))
        
        plt.tight_layout()
        plt.show()


# Crear instancia del ambiente
env = MazeEnv(ARCHIVO_LABERINTO)

print(f"\n✅ Ambiente creado correctamente")

### Prueba del Ambiente

Vamos a probar que el ambiente funciona correctamente ejecutando algunos pasos manualmente.

In [ ]:
# Probar reset
state = env.reset()
print(f"🔄 Estado después de reset(): {state}")
print(f"   Índice en Q-tabla: {env.state_to_index(state)}")

# Probar acciones desde el estado inicial
print(f"\n🎯 Probando acciones desde {state}:")
for action in range(NUM_ACCIONES):
    is_valid = env.is_valid_action(state, action)
    print(f"   Acción {action} ({ACCIONES[action]:10s}): {'✅ Válida' if is_valid else '❌ Inválida (pared/borde)'}")

# Ejecutar una acción válida
print(f"\n▶️  Ejecutando acción 3 (DERECHA):")
next_state, reward, done, info = env.step(3)
print(f"   Estado siguiente: {next_state}")
print(f"   Recompensa: {reward}")
print(f"   Episodio terminado: {done}")
print(f"   Info: {info}")

# Intentar una acción inválida (chocar con pared)
print(f"\n▶️  Intentando acción 0 (ARRIBA desde {next_state}):")
next_state2, reward2, done2, info2 = env.step(0)
print(f"   Estado siguiente: {next_state2}")
print(f"   Recompensa: {reward2}")
print(f"   Acción válida: {info2['valid_action']}")

## **Definición del agente**


### Clase QLearningAgent - Agente de Aprendizaje

Esta clase implementa el **algoritmo de Q-Learning** con política ε-greedy.

**Ecuación de actualización:**

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

**Separación de responsabilidades:**
- `MazeEnv` → Maneja el ambiente (transiciones, recompensas)
- `QLearningAgent` → Maneja el aprendizaje (Q-tabla, política)

In [ ]:
class QLearningAgent:
    """
    Agente de Q-Learning con política ε-greedy.
    
    Implementa el algoritmo off-policy Q-Learning para aprender
    la función Q óptima mediante exploración y explotación.
    """
    
    def __init__(self, num_states: int, num_actions: int, 
                 alpha: float = TASA_APRENDIZAJE,
                 gamma: float = FACTOR_DESCUENTO):
        """
        Inicializa el agente de Q-Learning.
        
        Args:
            num_states: Número de estados en el ambiente
            num_actions: Número de acciones posibles
            alpha: Tasa de aprendizaje (α)
            gamma: Factor de descuento (γ)
        """
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        
        # Inicializar Q-tabla con ceros
        # Forma: [num_states, num_actions]
        self.q_table = np.zeros((num_states, num_actions))
        
        print(f"🤖 Agente Q-Learning inicializado:")
        print(f"   - Q-tabla: {self.q_table.shape} (estados × acciones)")
        print(f"   - α (tasa aprendizaje): {self.alpha}")
        print(f"   - γ (factor descuento): {self.gamma}")
    
    def select_action(self, state_index: int, epsilon: float, 
                      valid_actions: List[int]) -> int:
        """
        Selecciona una acción usando política ε-greedy.
        
        Con probabilidad ε: explora (acción aleatoria válida)
        Con probabilidad 1-ε: explota (mejor Q-valor entre acciones válidas)
        
        Args:
            state_index: Índice del estado actual en la Q-tabla
            epsilon: Probabilidad de exploración (0 ≤ ε ≤ 1)
            valid_actions: Lista de acciones válidas desde el estado actual
            
        Returns:
            Acción seleccionada
        """
        if len(valid_actions) == 0:
            raise ValueError("No hay acciones válidas disponibles")
        
        # Exploración: acción aleatoria
        if np.random.random() < epsilon:
            return np.random.choice(valid_actions)
        
        # Explotación: mejor acción según Q-valores
        # Solo considerar acciones válidas
        q_values = self.q_table[state_index, valid_actions]
        best_action_idx = np.argmax(q_values)
        return valid_actions[best_action_idx]
    
    def update_q_value(self, state_index: int, action: int, 
                       reward: float, next_state_index: int, done: bool):
        """
        Actualiza el Q-valor usando la ecuación de Q-learning.
        
        Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]
        
        Args:
            state_index: Índice del estado actual
            action: Acción ejecutada
            reward: Recompensa recibida
            next_state_index: Índice del estado siguiente
            done: True si el episodio terminó
        """
        # Valor actual de Q(s,a)
        current_q = self.q_table[state_index, action]
        
        # Mejor Q-valor del siguiente estado: max_a' Q(s',a')
        if done:
            # Si el episodio terminó, no hay estado futuro
            max_next_q = 0.0
        else:
            max_next_q = np.max(self.q_table[next_state_index, :])
        
        # Ecuación de Q-learning
        # Temporal Difference (TD) target: r + γ·max Q(s',a')
        td_target = reward + self.gamma * max_next_q
        
        # TD error: δ = target - current
        td_error = td_target - current_q
        
        # Actualizar Q(s,a)
        new_q = current_q + self.alpha * td_error
        self.q_table[state_index, action] = new_q
    
    def get_greedy_action(self, state_index: int, valid_actions: List[int]) -> int:
        """
        Obtiene la mejor acción según la Q-tabla (sin exploración).
        
        Útil para evaluar el agente entrenado.
        
        Args:
            state_index: Índice del estado actual
            valid_actions: Lista de acciones válidas
            
        Returns:
            Mejor acción según Q-valores
        """
        if len(valid_actions) == 0:
            raise ValueError("No hay acciones válidas disponibles")
        
        q_values = self.q_table[state_index, valid_actions]
        best_action_idx = np.argmax(q_values)
        return valid_actions[best_action_idx]
    
    def save_q_table(self, filename: str):
        """
        Guarda la Q-tabla en un archivo .npy.
        
        Args:
            filename: Ruta del archivo donde guardar
        """
        np.save(filename, self.q_table)
        print(f"💾 Q-tabla guardada en: {filename}")
    
    def load_q_table(self, filename: str):
        """
        Carga una Q-tabla desde un archivo .npy.
        
        Args:
            filename: Ruta del archivo a cargar
        """
        self.q_table = np.load(filename)
        print("\n" + "="*60)
        print(f"📂 Q-tabla cargada desde: {filename}")
        print(f"   - Forma: {self.q_table.shape}")
        print("="*60 + "\n")
    
    def get_q_stats(self) -> dict:
        """
        Obtiene estadísticas de la Q-tabla para análisis.
        
        Returns:
            Diccionario con estadísticas
        """
        return {
            "q_mean": np.mean(self.q_table),
            "q_std": np.std(self.q_table),
            "q_min": np.min(self.q_table),
            "q_max": np.max(self.q_table),
            "non_zero": np.count_nonzero(self.q_table),
            "total": self.q_table.size
        }


# Crear instancia del agente
agent = QLearningAgent(env.num_states, NUM_ACCIONES)

print(f"\n✅ Agente creado correctamente")

# **3. ENTRENAMIENTO CON EPSILON DECAY**


## Entrenamiento
Ahora implementamos el bucle de entrenamiento que:
1. Ejecuta episodios de interacción ambiente-agente
2. Aplica **epsilon decay** para balancear exploración/explotación
3. Registra métricas para análisis

**Epsilon Decay:** $\varepsilon_{new} = \max(\varepsilon_{min}, \varepsilon_{old} \times decay)$

In [ ]:
def train_agent(env: MazeEnv, agent: QLearningAgent, 
                num_episodes: int = NUM_EPISODIOS,
                epsilon_inicial: float = EPSILON_INICIAL,
                epsilon_minimo: float = EPSILON_MINIMO,
                epsilon_decay: float = EPSILON_DECAY,
                verbose_freq: int = 100) -> dict:
    """
    Entrena al agente usando Q-Learning con epsilon decay.
    
    Args:
        env: Ambiente del laberinto
        agent: Agente de Q-Learning
        num_episodes: Número de episodios de entrenamiento
        epsilon_inicial: Valor inicial de epsilon
        epsilon_minimo: Valor mínimo de epsilon
        epsilon_decay: Factor de decaimiento (multiplicativo)
        verbose_freq: Frecuencia de impresión de progreso
        
    Returns:
        Diccionario con métricas del entrenamiento
    """
    # Inicializar epsilon y métricas
    epsilon = epsilon_inicial
    
    # Listas para almacenar métricas
    episode_rewards = []
    episode_lengths = []
    epsilon_history = []
    success_rate_window = []
    
    print(f"🎓 Iniciando entrenamiento...")
    print(f"   - Episodios: {num_episodes}")
    print(f"   - Epsilon: {epsilon_inicial} → {epsilon_minimo} (decay={epsilon_decay})")
    print(f"=" * 70)
    
    for episode in range(num_episodes):
        # Reiniciar ambiente
        state = env.reset()
        state_index = env.state_to_index(state)
        
        total_reward = 0
        done = False
        steps = 0
        
        # Ejecutar episodio
        while not done:
            # Obtener acciones válidas desde el estado actual
            valid_actions = [a for a in range(NUM_ACCIONES) 
                           if env.is_valid_action(state, a)]
            
            # Si no hay acciones válidas, el episodio termina
            if len(valid_actions) == 0:
                break
            
            # Seleccionar acción con política ε-greedy
            action = agent.select_action(state_index, epsilon, valid_actions)
            
            # Ejecutar acción
            next_state, reward, done, info = env.step(action)
            next_state_index = env.state_to_index(next_state)
            
            # Actualizar Q-valor
            agent.update_q_value(state_index, action, reward, next_state_index, done)
            
            # Acumular recompensa y pasos
            total_reward += reward
            steps += 1
            
            # Transición al siguiente estado
            state = next_state
            state_index = next_state_index
        
        # Registrar métricas del episodio
        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        epsilon_history.append(epsilon)
        
        # Calcular tasa de éxito (episodios que alcanzaron GOAL en últimos 100)
        reached_goal = (state == env.goal_state)
        success_rate_window.append(1 if reached_goal else 0)
        if len(success_rate_window) > 100:
            success_rate_window.pop(0)
        
        # Aplicar epsilon decay
        epsilon = max(epsilon_minimo, epsilon * epsilon_decay)
        
        # Imprimir progreso
        if (episode + 1) % verbose_freq == 0:
            avg_reward = np.mean(episode_rewards[-verbose_freq:])
            avg_steps = np.mean(episode_lengths[-verbose_freq:])
            success_rate = np.mean(success_rate_window) * 100
            
            print(f"Episodio {episode + 1:4d}/{num_episodes} | "
                  f"ε={epsilon:.3f} | "
                  f"Reward promedio={avg_reward:7.2f} | "
                  f"Pasos promedio={avg_steps:5.1f} | "
                  f"Éxito últimos 100={success_rate:5.1f}%")
    
    print(f"=" * 70)
    print(f"✅ Entrenamiento completado")
    
    # Retornar métricas
    return {
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "epsilon_history": epsilon_history,
        "final_epsilon": epsilon,
        "final_success_rate": np.mean(success_rate_window) * 100
    }


if VALIDAR_Q_TABLA == False:
    print(f"\n📝 Estado: Q-tabla no encontrada - Se entrenará un nuevo agente")
    # Entrenar al agente
    start_time = time.time()

    metrics = train_agent(env, agent, 
                        num_episodes=NUM_EPISODIOS,
                        epsilon_inicial=EPSILON_INICIAL,
                        epsilon_minimo=EPSILON_MINIMO,
                        epsilon_decay=EPSILON_DECAY,
                        verbose_freq=100)

    elapsed_time = time.time() - start_time

    # Guardar la Q-tabla entrenada
    agent.save_q_table(ARCHIVO_Q_TABLE)


    # ======================================================
    # 📈 RESUMEN DEL ENTRENAMIENTO
    # ======================================================
    print("\n" + "="*60)
    print("📈 RESUMEN DEL ENTRENAMIENTO")
    print("="*60)

    # Tiempo de entrenamiento
    print(f"⏱️  Tiempo total : {elapsed_time:6.2f} segundos "
        f"({elapsed_time/60:6.2f} minutos)")

    # Métrica principal
    print(f"🏁 Tasa de éxito (últimos 100 episodios): "
        f"{metrics['final_success_rate']:5.1f}%")

    # ======================================================
    # 🧠 ESTADÍSTICAS DE LA Q-TABLE
    # ======================================================
    q_stats = agent.get_q_stats()

    print("\n" + "="*60)
    print("🧠 ESTADÍSTICAS DE LA Q-TABLE")
    print("="*60)
    print(f"📐 Forma              : {agent.q_table.shape}")
    print(f"📊 Q promedio         : {q_stats['q_mean']:7.3f}")
    print(f"📊 Q desviación std   : {q_stats['q_std']:7.3f}")
    print(f"🔻 Q mínimo           : {q_stats['q_min']:7.3f}")
    print(f"🔺 Q máximo           : {q_stats['q_max']:7.3f}")

    porc_no_cero = 100 * q_stats['non_zero'] / q_stats['total']
    print(f"🔢 Valores no cero    : {q_stats['non_zero']}/{q_stats['total']} "
        f"({porc_no_cero:5.1f}%)")
    print("="*60 + "\n")

else:
    print(f"\n🎯 Estado: Q-tabla encontrada - Se puede cargar para evaluación")
    print(f"\n🎯 No se entrenará el modelo")

## Evaluación del entrenamiento

Analicemos cómo evolucionó el aprendizaje del agente.

In [ ]:


def diagnostic_training(env: MazeEnv, agent: QLearningAgent,
                        num_episodes: int = 100,
                        epsilon_inicial: float = EPSILON_INICIAL,
                        epsilon_minimo: float = EPSILON_MINIMO,
                        epsilon_decay: float = EPSILON_DECAY) -> dict:
        """
        Entrena al agente con diagnóstico detallado.
        """
        epsilon = epsilon_inicial
        
        # Estructuras para almacenar datos diagnósticos
        episodes_data = []
        state_visits = np.zeros((env.n_filas, env.n_columnas))
        q_evolution = []
        
        print("🔬 ENTRENAMIENTO DIAGNÓSTICO")
        print(f"   - Episodios: {num_episodes}")
        print(f"   - Epsilon: {epsilon_inicial} → {epsilon_minimo}")
        print("=" * 80)
        
        for episode in range(num_episodes):
            state = env.reset()
            state_index = env.state_to_index(state)
            
            total_reward = 0
            done = False
            steps = 0
            exploration_count = 0
            exploitation_count = 0
            episode_path = [state]
            
            while not done and steps < MAX_PASOS_POR_EPISODIO:
                valid_actions = [a for a in range(NUM_ACCIONES) 
                                if env.is_valid_action(state, a)]
                if not valid_actions:
                    break
                
                # Selección de acción
                if np.random.random() < epsilon:
                    action = np.random.choice(valid_actions)
                    exploration_count += 1
                else:
                    q_values = agent.q_table[state_index, valid_actions]
                    action = valid_actions[int(np.argmax(q_values))]
                    exploitation_count += 1
                
                next_state, reward, done, info = env.step(action)
                next_state_index = env.state_to_index(next_state)
                
                agent.update_q_value(state_index, action, reward, next_state_index, done)
                state_visits[state[0], state[1]] += 1
                
                total_reward += reward
                steps += 1
                episode_path.append(next_state)
                
                state = next_state
                state_index = next_state_index
            
            reached_goal = (state == env.goal_state)
            episodes_data.append({
                "episode": episode + 1,
                "reward": total_reward,
                "steps": steps,
                "exploration_steps": exploration_count,
                "exploitation_steps": exploitation_count,
                "reached_goal": reached_goal,
                "epsilon": epsilon,
                "path": episode_path
            })
            
            q_stats = agent.get_q_stats()
            non_zero = q_stats['non_zero']
            total_q = q_stats['total']
            q_evolution.append({
                "episode": episode + 1,
                "mean": q_stats['q_mean'],
                "std": q_stats['q_std'],
                "min": q_stats['q_min'],
                "max": q_stats['q_max'],
                "non_zero": non_zero,
                "total": total_q,
                "non_zero_ratio": non_zero / total_q if total_q else 0.0
            })
            
            epsilon = max(epsilon_minimo, epsilon * epsilon_decay)
            
            if (episode + 1) % 10 == 0:
                last_10 = episodes_data[-10:]
                success_count = sum(ep['reached_goal'] for ep in last_10)
                success_rate = success_count * 10.0  # /10*100
                rewards_10 = [ep['reward'] for ep in last_10]
                explore_10 = [ep['exploration_steps'] for ep in last_10]
                exploit_10 = [ep['exploitation_steps'] for ep in last_10]
                
                avg_reward = float(np.mean(rewards_10))
                avg_explore = float(np.mean(explore_10))
                avg_exploit = float(np.mean(exploit_10))
                
                print(f"Ep {episode+1:3d}/{num_episodes} | "
                    f"ε={epsilon:.3f} | "
                    f"Éxito={success_rate:5.1f}% | "
                    f"Reward={avg_reward:7.1f} | "
                    f"Explora={avg_explore:4.1f} | "
                    f"Explota={avg_exploit:4.1f}")
        
        print("=" * 80)
        print("✅ Entrenamiento diagnóstico completado")
        
        return {
            "episodes_data": episodes_data,
            "state_visits": state_visits,
            "q_evolution": q_evolution,
            "final_epsilon": epsilon
        }


    # ----------------- Helpers reutilizables -----------------
def moving_average(data, window):
        if len(data) < window:
            return None
        kernel = np.ones(window) / window
        return np.convolve(data, kernel, mode='valid')

def _compute_block_stats(bloque):
    success = sum(1 for ep in bloque if ep["reached_goal"])
    rewards_block = [ep["reward"] for ep in bloque]
    expl_block = [ep["exploration_steps"] for ep in bloque]
    exploit_block = [ep["exploitation_steps"] for ep in bloque]
    return {
        "success": success,
        "reward_mean": float(np.mean(rewards_block)),
        "expl_mean": float(np.mean(expl_block)),
        "exploit_mean": float(np.mean(exploit_block))
    }

def print_ep_stats(nombre, bloque):
    stats = _compute_block_stats(bloque)
    print(f"\n📌 {nombre}")
    print(f"   🟢 Éxito               : {stats['success']}/10")
    print(f"   💰 Reward promedio     : {stats['reward_mean']:6.1f}")
    print(f"   🧭 Exploración promedio: {stats['expl_mean']:6.1f} pasos")
    print(f"   🎯 Explotación prom.   : {stats['exploit_mean']:6.1f} pasos")


    # ===================== EJECUCIÓN ENTRENAMIENTO =====================
if VALIDAR_Q_TABLA == False:
    print(f"\n📝 Estado: Q-tabla no encontrada - Se evaluará el nuevo agente")

    diagnostic_results = diagnostic_training(
        env=env,
        agent=agent,
        num_episodes=100,
        epsilon_inicial=EPSILON_INICIAL,
        epsilon_minimo=EPSILON_MINIMO,
        epsilon_decay=EPSILON_DECAY
    )

    # ===================== VISUALIZACIÓN =====================

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    episodes_data = diagnostic_results['episodes_data']
    q_evolution = diagnostic_results['q_evolution']
    state_visits = diagnostic_results['state_visits']

    episodes = [ep['episode'] for ep in episodes_data]
    explore_steps = [ep['exploration_steps'] for ep in episodes_data]
    exploit_steps = [ep['exploitation_steps'] for ep in episodes_data]
    rewards = [ep['reward'] for ep in episodes_data]
    successes = [1 if ep['reached_goal'] else 0 for ep in episodes_data]
    episode_lengths = [ep['steps'] for ep in episodes_data]
    non_zero_ratios = [100 * q['non_zero_ratio'] for q in q_evolution]

    cumulative_success = np.cumsum(successes)
    success_rate = [100 * cumulative_success[i] / (i + 1) for i in range(len(successes))]

    window = 10

    # 1. Exploración vs Explotación por episodio
    ax1 = axes[0, 0]
    ax1.plot(episodes, explore_steps, 'b-', alpha=0.7, label='Exploración')
    ax1.plot(episodes, exploit_steps, 'r-', alpha=0.7, label='Explotación')
    ax1.set_xlabel('Episodio', fontsize=11)
    ax1.set_ylabel('Número de Pasos', fontsize=11)
    ax1.set_title('Exploración vs Explotación por Episodio', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. Frecuencia de Visitas a Estados
    ax2 = axes[0, 1]
    im = ax2.imshow(state_visits, cmap='YlOrRd', aspect='auto')
    ax2.set_xlabel('Columna', fontsize=11)
    ax2.set_ylabel('Fila', fontsize=11)
    ax2.set_title('Frecuencia de Visitas a Estados', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax2, label='Visitas')

    start_f, start_c = env.start_state
    goal_f, goal_c = env.goal_state
    ax2.scatter(start_c, start_f, marker='o', s=200, facecolors='none',
                edgecolors='blue', linewidths=3, label='START')
    ax2.scatter(goal_c, goal_f, marker='*', s=300, color='green',
                edgecolors='black', linewidths=1, label='GOAL')
    ax2.legend()

    # 3. Recompensas por Episodio
    ax3 = axes[0, 2]
    ax3.plot(episodes, rewards, 'orange', alpha=0.6, label='Reward')
    moving_avg = moving_average(rewards, window)
    if moving_avg is not None:
        ax3.plot(range(window, len(rewards) + 1), moving_avg, 'r-', linewidth=2,
                label=f'Media móvil ({window})')
    ax3.set_xlabel('Episodio', fontsize=11)
    ax3.set_ylabel('Reward Total', fontsize=11)
    ax3.set_title('Recompensas por Episodio', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Tasa de Éxito Acumulada
    ax4 = axes[1, 0]
    ax4.plot(episodes, success_rate, 'purple', linewidth=2)
    ax4.set_xlabel('Episodio', fontsize=11)
    ax4.set_ylabel('Tasa de Éxito (%)', fontsize=11)
    ax4.set_title('Tasa de Éxito Acumulada', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim([0, 105])

    # 5. Convergencia de la Q-tabla
    ax5 = axes[1, 1]
    ax5.plot(episodes, non_zero_ratios, 'teal', linewidth=2)
    ax5.set_xlabel('Episodio', fontsize=11)
    ax5.set_ylabel('% Q-valores ≠ 0', fontsize=11)
    ax5.set_title('Convergencia de la Q-tabla', fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3)
    ax5.set_ylim([0, 105])

    # 6. Longitud de Episodios
    ax6 = axes[1, 2]
    ax6.plot(episodes, episode_lengths, 'g-', alpha=0.6, label='Pasos por episodio')
    moving_avg_len = moving_average(episode_lengths, window)
    if moving_avg_len is not None:
        ax6.plot(range(window, len(episode_lengths) + 1), moving_avg_len,
                'darkgreen', linewidth=2, label=f'Media móvil ({window})')
    ax6.set_xlabel('Episodio', fontsize=11)
    ax6.set_ylabel('Número de Pasos', fontsize=11)
    ax6.set_title('Longitud de Episodios', fontsize=12, fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ===================== RESUMEN TEXTO =====================

    print("\n" + "="*70)
    print("📊 RESUMEN DE DATOS DIAGNÓSTICOS")
    print("="*70)

    visited_states = np.count_nonzero(state_visits)
    visited_ratio = 100 * visited_states / env.num_states

    print(f"🗂️  Episodios registrados : {len(episodes_data)}")
    print(f"🧭 Estados visitados      : {visited_states} / {env.num_states} "
        f"({visited_ratio:.1f}%)")
    print(f"📈 Puntos evolución Q      : {len(q_evolution)}")

    print("\n" + "="*70)
    print("🔍 ANÁLISIS DE CONVERGENCIA")
    print("="*70)

    first_10 = episodes_data[:10]
    last_10  = episodes_data[-10:]

    print_ep_stats("Primeros 10 episodios", first_10)
    print_ep_stats("Últimos 10 episodios", last_10)

    print("\n" + "="*70)
    print("💡 INTERPRETACIÓN")
    print("="*70)

    success_first = sum(1 for ep in first_10 if ep["reached_goal"])
    success_last  = sum(1 for ep in last_10 if ep["reached_goal"])
    improvement   = success_last - success_first

    if success_last == 0:
        print("⚠️  El agente NO muestra señales de aprendizaje (0% éxito en los últimos 10 episodios).")
        print("\nPosibles causas:")
        print("   1️⃣  MAX_PASOS demasiado bajo para alcanzar la meta.")
        print("   2️⃣  Recompensas mal calibradas (penalización excesiva o camino costoso).")
        print("   3️⃣  α (learning rate) o γ (discount) inadecuados.")
        print("   4️⃣  Laberinto con muchos bloqueos o trampas estructurales.")
    else:
        print(f"✅ El agente muestra mejora: {success_first} → {success_last} episodios exitosos "
            f"(Δ = {improvement}).")
    print("\n" + "="*70 + "\n")
else:
    print(f"\n🎯 Estado: Q-tabla encontrada - Se puede cargar para evaluación")
    print(f"\n🎯 No se entrenará el modelo")

# **4. CARGAR Q-TABLA PARA EJECUTAR EL AGENTE ENTRENADO**
### Evaluación del Agente (Modo Greedy)

Probemos el agente entrenado ejecutando episodios SIN exploración (ε=0), usando solo la política aprendida.

In [ ]:

def evaluate_agent_window(env: MazeEnv, agent: QLearningAgent, 
                         num_episodes: int = 3, 
                         delay_seconds: float = 0.3) -> dict:
    """
    Evalúa al agente con visualización en VENTANA EXTERNA.
    Usa backend TkAgg para mostrar la animación en una ventana separada.
    
    Args:
        env: Ambiente del laberinto
        agent: Agente entrenado
        num_episodes: Número de episodios a visualizar
        delay_seconds: Tiempo de espera entre movimientos (segundos)
        
    Returns:
        Diccionario con resultados de la evaluación
    """
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle, Circle
    import time
    
    # Cambiar backend a TkAgg para ventanas externas
    matplotlib.use('TkAgg')
    print("=" * 70 + "\n")
    print(f"🎬 Evaluando agente con visualización en VENTANA EXTERNA")
    print(f"   - Episodios: {num_episodes}")
    print(f"   - Velocidad: {delay_seconds}s por movimiento")
    print(f"   - Backend: {matplotlib.get_backend()}")
    print("\n" + "=" * 70)
    print("⚠️  IMPORTANTE: La ventana se abrirá automáticamente. No la cierres manualmente.")
    print("=" * 70)
    
    eval_rewards = []
    eval_lengths = []
    eval_successes = []
    
    for episode in range(num_episodes):
        print(f"\n▶️  Episodio {episode + 1}/{num_episodes}")
        
        state = env.reset()
        state_index = env.state_to_index(state)
        
        total_reward = 0
        done = False
        steps = 0
        path = [state]
        
        # Crear figura para este episodio (ventana nueva)
        plt.ion()  # Activar modo interactivo
        fig, ax = plt.subplots(figsize=(10, 10))
        fig.canvas.manager.set_window_title(f'Q-Learning Maze - Episodio {episode + 1}')
        
        ax.set_xlim(0, env.n_columnas)
        ax.set_ylim(0, env.n_filas)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_xlabel('Columnas', fontsize=12)
        ax.set_ylabel('Filas', fontsize=12)
        ax.set_title(f'Episodio {episode + 1} - Paso 0', fontsize=14, fontweight='bold')
        
        # Dibujar paredes del laberinto
        for (celda1, celda2) in env.paredes:
            f1, c1 = celda1
            f2, c2 = celda2
            
            if f1 == f2:  # Pared vertical
                x = [max(c1, c2), max(c1, c2)]
                y = [f1, f1 + 1]
            else:  # Pared horizontal
                x = [c1, c1 + 1]
                y = [max(f1, f2), max(f1, f2)]
            
            ax.plot(x, y, 'k-', linewidth=2)
        
        # Dibujar bordes exteriores
        ax.plot([0, env.n_columnas], [0, 0], 'k-', linewidth=3)
        ax.plot([0, env.n_columnas], [env.n_filas, env.n_filas], 'k-', linewidth=3)
        ax.plot([0, 0], [0, env.n_filas], 'k-', linewidth=3)
        ax.plot([env.n_columnas, env.n_columnas], [0, env.n_filas], 'k-', linewidth=3)
        
        # Marcar START y GOAL
        start_f, start_c = env.start_state
        goal_f, goal_c = env.goal_state
        
        start_patch = Rectangle((start_c, start_f), 1, 1, 
                                facecolor='red', alpha=0.3, label='START')
        goal_patch = Rectangle((goal_c, goal_f), 1, 1, 
                               facecolor='green', alpha=0.3, label='GOAL')
        ax.add_patch(start_patch)
        ax.add_patch(goal_patch)
        
        # Crear objeto para la trayectoria (línea que se va dibujando)
        trajectory_line, = ax.plot([], [], 'b-', linewidth=2, alpha=0.5, label='Trayectoria')
        
        # Crear objeto para el agente (círculo amarillo)
        agent_circle = Circle((start_c + 0.5, start_f + 0.5), 0.3, 
                             color='yellow', ec='black', linewidth=2, 
                             label='Agente', zorder=10)
        ax.add_patch(agent_circle)
        
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_xticks(range(env.n_columnas + 1))
        ax.set_yticks(range(env.n_filas + 1))
        
        plt.tight_layout()
        plt.show(block=False)  # Mostrar sin bloquear
        plt.pause(0.8)  # Pausa inicial para ver el estado inicial
        
        # Ejecutar episodio con visualización
        while not done and steps < MAX_PASOS_POR_EPISODIO:
            # Obtener acciones válidas
            valid_actions = [a for a in range(NUM_ACCIONES) 
                           if env.is_valid_action(state, a)]
            
            if len(valid_actions) == 0:
                break
            
            # Seleccionar mejor acción (greedy)
            action = agent.get_greedy_action(state_index, valid_actions)
            
            # Ejecutar acción
            next_state, reward, done, info = env.step(action)
            next_state_index = env.state_to_index(next_state)
            
            total_reward += reward
            steps += 1
            path.append(next_state)
            
            # Actualizar visualización
            # 1. Actualizar trayectoria
            path_x = [c + 0.5 for f, c in path]
            path_y = [f + 0.5 for f, c in path]
            trajectory_line.set_data(path_x, path_y)
            
            # 2. Actualizar posición del agente
            curr_f, curr_c = next_state
            agent_circle.center = (curr_c + 0.5, curr_f + 0.5)
            
            # 3. Actualizar título con información
            action_name = ACCIONES[action]
            ax.set_title(
                f'Episodio {episode + 1} - Paso {steps} | '
                f'Acción: {action_name} | Reward: {reward:.1f}',
                fontsize=14, fontweight='bold'
            )
            
            # 4. Redibujar
            fig.canvas.draw()
            fig.canvas.flush_events()
            time.sleep(delay_seconds)
            
            state = next_state
            state_index = next_state_index
        
        # Registrar resultados
        eval_rewards.append(total_reward)
        eval_lengths.append(steps)
        reached_goal = (state == env.goal_state)
        eval_successes.append(1 if reached_goal else 0)
        
        # Mostrar resultado final del episodio
        status = "✅ ¡META ALCANZADA!" if reached_goal else "❌ No alcanzó la meta"
        final_title = (f'Episodio {episode + 1} FINALIZADO - {status}\n'
                      f'Pasos: {steps} | Reward total: {total_reward:.1f}')
        ax.set_title(final_title, fontsize=14, fontweight='bold')
        
        # Cambiar color del agente según resultado
        agent_circle.set_color('lime' if reached_goal else 'red')
        
        fig.canvas.draw()
        time.sleep(2.5)  # Pausa más larga al final para ver el resultado
        
        plt.close(fig)
        
        # Imprimir resumen del episodio
        print(f"   {status}")
        print(f"   Pasos: {steps} | Reward: {total_reward:.1f}")
    
    # Desactivar modo interactivo
    plt.ioff()
    
    print("\n" + "=" * 70)
    
    # Estadísticas finales
    success_rate = np.mean(eval_successes) * 100
    avg_reward = np.mean(eval_rewards)
    avg_steps = np.mean(eval_lengths)
    
    print(f"\n📊 RESULTADOS FINALES:")
    print(f"   - Tasa de éxito: {success_rate:.1f}% ({sum(eval_successes)}/{num_episodes})")
    print(f"   - Recompensa promedio: {avg_reward:.1f}")
    print(f"   - Pasos promedio: {avg_steps:.1f}")
    
    # Restaurar backend inline para Jupyter
    matplotlib.use('module://matplotlib_inline.backend_inline')
    print(f"\n✅ Backend restaurado a: {matplotlib.get_backend()}")
    
    return {
        "rewards": eval_rewards,
        "lengths": eval_lengths,
        "successes": eval_successes,
        "success_rate": success_rate
    }

VALIDAR_Q_TABLA = validar_q_table(ARCHIVO_Q_TABLE)

if VALIDAR_Q_TABLA:
    print(f"\n🎯 Estado: Q-tabla encontrada - Se carga para evaluación")

    print(" Función evaluate_agent_window() creada.")
    print("   Muestra la animación en una VENTANA EXTERNA separada.")
    print("   Usa backend TkAgg para ventanas interactivas.")
    print("   ⚠️  Nota: No cierres manualmente la ventana durante la ejecución.")


    # Cargar agente entrenado desde archivo y evaluar con visualización en ventana externa
    agent_loaded = QLearningAgent(env.num_states, NUM_ACCIONES)
    agent_loaded.load_q_table(ARCHIVO_Q_TABLE)
    eval_loaded = evaluate_agent_window(
        env=env,
        agent=agent_loaded,
        num_episodes=1,
        delay_seconds=0.1
    )

else:
    print(f"\n📝 Estado: Q-tabla no encontrada - Se requiere entrenar un nuevo agente")